# Entrenar el modelo pequeño (LoRA) en Colab

Este notebook envuelve **el mismo script** que corre en Vertex AI (`train_lora.py`): LoRA con TRL + PEFT sobre Gemma, merge, GGUF Q4_K_M con llama.cpp y Modelfile para Ollama.

- Runtime: GPU (L4 o A100 en Colab Pro / Colab Enterprise; T4 funciona pero más lento y sin bf16).
- Pesos base: bucket público de Vertex Model Garden (no hace falta token de Hugging Face).
- Archivos: sube `train/train_lora.py` y los datos.
- Datos: `data/train/chat.jsonl`, generado en el repo con `make train-data (o el script de la app que genere chat.jsonl)` (súbelo al panel de archivos o usa una ruta `gs://`).

In [ ]:
# Parámetros (los mismos env vars que el repo)
TAG = "<app>-YYYY-MM"            # tag con la fecha de los datos
GEMMA_BASE_MODEL = "gemma-4-E2B-it"
GEMMA_WEIGHTS_URI = "gs://vertex-model-garden-public-us/gemma4"
DATA = "chat.jsonl"                   # o gs://ai-experiments-487722-train-eu/<app>/data/<tag>/train.jsonl
OUTPUT_URI = ""                       # gs://<bucket-de-modelos> para publicar; vacío = local
LLAMA_CPP_TAG = "b11292"

In [ ]:
# Colab Enterprise ya tiene credenciales; en Colab normal, autentícate para leer/escribir GCS.
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

In [ ]:
# Mismas versiones que la imagen de Vertex AI (train/requirements.txt).
!pip -q install transformers==5.17.0 peft==0.21.1 trl==1.14.1 accelerate==1.15.0 datasets==5.0.1 google-cloud-storage==3.15.1 sentencepiece==0.2.2
import os
assert os.path.exists("train_lora.py"), "Sube train/train_lora.py del repo al panel de archivos"

In [ ]:
# llama.cpp: el convertidor HF -> GGUF y el cuantizador (compilado solo para CPU)
!git clone -q --depth 1 --branch {LLAMA_CPP_TAG} https://github.com/ggml-org/llama.cpp /content/llama.cpp || true
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_NATIVE=OFF -DLLAMA_CURL=OFF -DBUILD_SHARED_LIBS=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /dev/null && cp /content/llama.cpp/build/bin/llama-quantize /usr/local/bin/

## Entrenar

r=16, alpha=32, lr 2e-4, 3 épocas. Solo aprende el decodificador de texto; las torres de visión y audio de Gemma quedan congeladas.

In [ ]:
!python train_lora.py --data {DATA} --tag {TAG} --base-model {GEMMA_BASE_MODEL} \
    --weights-uri {GEMMA_WEIGHTS_URI} --output-uri "{OUTPUT_URI}" \
    --llama-cpp /content/llama.cpp --work-dir /content/work

## Resultado

En `/content/work/out/`: `<tag>-q4_k_m.gguf`, `<tag>-f16.gguf`, `Modelfile`, el adaptador LoRA y `metrics.json`. En la laptop: `make pull-model TAG=<tag>` (si publicaste a GCS) o copia el GGUF + Modelfile y corre `ollama create <tag> -f Modelfile`.

In [ ]:
import json; print(json.dumps(json.load(open("/content/work/out/metrics.json")), indent=2))